<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Chapter 3 Exercise solutions

# Exercise 3.1

In [1]:
import torch

# 用 6 个 3 维向量表示一句话中的 6 个 token。
# 每一行可以理解为一个词的“坐标”，模型会基于这些坐标计算词与词的关系。
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

# d_in 是输入向量维度，d_out 是注意力层输出的向量维度。
d_in, d_out = 3, 2
print(inputs)

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])


In [2]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_out = d_out
        # 这里直接把 W_query、W_key、W_value 定义为可训练参数矩阵。
        # 它们的作用是把输入 token 向量投影成 query、key、value。
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        # x 的形状是 (token 数量, 输入维度)。矩阵乘法后得到每个 token 的 Q/K/V 表示。
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        # query 与 key 做点积得到注意力分数 omega；分数越大，表示两个 token 越相关。
        attn_scores = queries @ keys.T # omega
        # 除以 key 维度的平方根可以让分数尺度更稳定，再用 softmax 转成权重。
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        # 用注意力权重对 value 加权求和，得到每个 token 的上下文向量。
        context_vec = attn_weights @ values
        return context_vec

# 固定随机种子，方便多次运行时得到相同的初始化结果。
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)

In [3]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_out = d_out
        # nn.Linear 等价于一个带权重矩阵的线性投影层。
        # bias=False 表示这里只保留权重矩阵，便于和 v1 的手写矩阵形式对比。
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key   = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)

    def forward(self, x):
        # Linear 层会自动完成 x @ weight.T，所以写法比 v1 更贴近常见 PyTorch 风格。
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # 计算注意力分数，并沿每一行做 softmax，让每个 token 的注意力权重和为 1。
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=1)

        # 权重乘以 value，得到融合了其他 token 信息的上下文向量。
        context_vec = attn_weights @ values
        return context_vec

# 使用同样的随机种子初始化，方便和 SelfAttention_v1 对照。
torch.manual_seed(123)
sa_v2 = SelfAttention_v2(d_in, d_out)

In [4]:
# v1 的参数形状是 (d_in, d_out)，而 nn.Linear 内部保存的是 (d_out, d_in)。
# 因此这里用 .T 转置，把 v2 的权重拷贝到 v1，方便比较两个实现是否等价。
sa_v1.W_query = torch.nn.Parameter(sa_v2.W_query.weight.T)
sa_v1.W_key = torch.nn.Parameter(sa_v2.W_key.weight.T)
sa_v1.W_value = torch.nn.Parameter(sa_v2.W_value.weight.T)

In [5]:
# 运行手写参数矩阵版本的自注意力。
sa_v1(inputs)

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)

In [6]:
# 运行 nn.Linear 版本的自注意力；和上一个单元输出应保持一致。
sa_v2(inputs)

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)

# Exercise 3.2

If we want to have an output dimension of 2, as earlier in single-head attention, we can have to change the projection dimension `d_out` to 1:

```python
torch.manual_seed(123)

d_out = 1
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)
```

```
tensor([[[-9.1476e-02,  3.4164e-02],
         [-2.6796e-01, -1.3427e-03],
         [-4.8421e-01, -4.8909e-02],
         [-6.4808e-01, -1.0625e-01],
         [-8.8380e-01, -1.7140e-01],
         [-1.4744e+00, -3.4327e-01]],

        [[-9.1476e-02,  3.4164e-02],
         [-2.6796e-01, -1.3427e-03],
         [-4.8421e-01, -4.8909e-02],
         [-6.4808e-01, -1.0625e-01],
         [-8.8380e-01, -1.7140e-01],
         [-1.4744e+00, -3.4327e-01]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])
```

# Exercise 3.3

```python
context_length = 1024
d_in, d_out = 768, 768
num_heads = 12

mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads)
```

Optionally, the number of parameters is as follows:

```python
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

count_parameters(mha)
```

```
2360064  # (2.36 M)
```

The GPT-2 model has 117M parameters in total, but as we can see, most of its parameters are not in the multi-head attention module itself.